<a href="https://colab.research.google.com/github/SimonHtet/Hotel/blob/main/exceltis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [191]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [192]:
import os
for root, dirs, files in os.walk('/content/drive/MyDrive'):
    for file in files:
        if 'Exeltis' in file:
            print(os.path.join(root, file))

/content/drive/MyDrive/Exceltis/Exeltis_Data_Engineer_Business_Case_Raw_Data.csv


In [193]:
# == section 2: profile/explore ==
# markdown cell above this ##2.profiling -finding data quality issues

#structure: are the data types right?--
df.info()
df.head()

#-- whats missing?
print(df.isnull().sum())
print()
print(df['country'].value_counts())
print(df['client_segment'].value_counts())
print('Full duplicate rows:', df.duplicated().sum())
print('Duplicate order_id:', df['order_id'].duplicated().sum())

<class 'pandas.core.frame.DataFrame'>
Index: 2498 entries, 0 to 2499
Data columns (total 18 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   order_id             2498 non-null   object        
 1   order_date           2497 non-null   datetime64[ns]
 2   customer_name        2498 non-null   object        
 3   client_segment       2498 non-null   object        
 4   client_city          2498 non-null   object        
 5   country              2498 non-null   object        
 6   client_singup_date   2479 non-null   datetime64[ns]
 7   product_name         2498 non-null   object        
 8   quantity             2498 non-null   int64         
 9   unit_price           2498 non-null   float64       
 10  unit_cost            2465 non-null   float64       
 11  discount_pct         2497 non-null   float64       
 12  product_category     2498 non-null   object        
 13  product_subcategory  2498 non-null   o

In [194]:
df[['quantity','unit_price','unit_cost','discount_pct']].describe()

,quantity,unit_price,unit_cost,discount_pct
count,2498.000000,2498.000000,2465.000000,2497.000000
mean,50.023619,38.227862,19.654921,0.171802
std,28.819322,20.892637,11.438005,0.101184
min,1.000000,2.010000,1.580000,0.000000
25%,24.000000,19.915000,10.660000,0.080000
50%,50.000000,37.965000,17.810000,0.170000
75%,75.000000,55.880000,29.070000,0.260000
max,100.000000,74.990000,39.970000,0.350000


In [195]:
#some unit cost are negative
import numpy as np

df.loc[df['unit_cost'] <= 0, 'unit_cost'] = np.nan

In [196]:
print('Rows where cost == price :', (df['unit_cost'] == df['unit_cost']).sum())
print('Rows with inactive product sold :', (df['product_active'] == False).sum())

Rows where cost == price : 2465
Rows with inactive product sold : 508


In [197]:
print(df['sales_channel'].value_counts())
print(df[['order_date','client_singup_date']].head())

sales_channel
Online       845
Phone        842
Sales Rep    811
Name: count, dtype: int64
  order_date client_singup_date
0 2024-09-13         2022-04-05
1 2023-01-27         2021-06-25
2 2023-03-19         2024-11-02
3 2024-09-06         2022-12-26
4 2024-01-15         2024-12-26


In [198]:
def parse_mixed_dates(val):
    try:
        return pd.Timestamp('1899-12-30') + pd.Timedelta(days=int(float(val)))
    except:
        pass
    for fmt in ('%d/%m/%Y', '%m/%d/%Y', '%Y-%m-%d', '%Y/%m/%d'):
        try:
            return pd.to_datetime(val, format=fmt)
        except:
            pass
    return pd.NaT

df['client_singup_date'] = df['client_singup_date'].apply(parse_mixed_dates).dt.strftime('%Y/%m/%d')
print(df['client_singup_date'].isna().sum())
print(df[['order_date','client_singup_date']].head())

19
  order_date client_singup_date
0 2024-09-13         2022/04/05
1 2023-01-27         2021/06/25
2 2023-03-19         2024/11/02
3 2024-09-06         2022/12/26
4 2024-01-15         2024/12/26


In [199]:
print(df['order_date'].iloc[80])

2024-07-04 00:00:00


In [200]:
df[df['order_date'] == '31-02-2024']

,order_id,order_date,customer_name,client_segment,client_city,country,client_singup_date,product_name,quantity,unit_price,unit_cost,discount_pct,product_category,product_subcategory,supplier_name,product_active,sales_channel,sales_rep


In [201]:
df['order_date'] = pd.to_datetime(df['order_date'], format='mixed', errors='coerce').dt.strftime('%Y/%m/%d')
print(df['order_date'].isna().sum())  # see how many became NaT

1


In [202]:
df[df.duplicated(keep=False)] #finding duplicated rows

,order_id,order_date,customer_name,client_segment,client_city,country,client_singup_date,product_name,quantity,unit_price,unit_cost,discount_pct,product_category,product_subcategory,supplier_name,product_active,sales_channel,sales_rep


In [203]:
df = df.drop_duplicates() #drop duplicates

In [204]:
print('duplicate order_ids:', df['order_id'].duplicated().sum())
print('rows now:',len(df))


duplicate order_ids: 0
rows now: 2498


In [205]:
df[df['discount_pct']>1]

,order_id,order_date,customer_name,client_segment,client_city,country,client_singup_date,product_name,quantity,unit_price,unit_cost,discount_pct,product_category,product_subcategory,supplier_name,product_active,sales_channel,sales_rep


In [206]:
df.loc[df['discount_pct']>1, 'discount_pct'] = pd.NA

In [207]:
print('discounts > 1 now:', (df['discount_pct']>1).sum())

discounts > 1 now: 0


In [208]:
df = df.copy()
df['country'] = df['country'].replace('0', 'Unknown')
df['client_segment'] = df['client_segment'].replace('0', 'Unknown')
print(df['country'].value_counts())
print(df['client_segment'].value_counts())

country
France      729
Portugal    674
Italy       588
Spain       506
Unknown       1
Name: count, dtype: int64
client_segment
Distributor    826
Retail         562
Hospitality    560
Supermarket    532
Unknown         18
Name: count, dtype: int64


In [209]:
print(df.shape)
print(df['order_date'].dtype)
print((df['country']=='0').sum())

(2498, 18)
object
0


In [210]:
import pandas as pd
import numpy as mp

print('row:', len(df))
print('dup order_id:', df['order_id'].duplicated().sum())
print('min quantity:', df['quantity'].min())
print('discount >1:', (df['discount_pct']>1).sum())
print("country '0':", (df['country'] == '0').sum(),"segment '0':",(df['client_segment']=='0').sum())

row: 2498
dup order_id: 0
min quantity: 1
discount >1: 0
country '0': 0 segment '0': 0


In [211]:
# some product categories is 0 or null
cat_lookup = (df.loc[df['product_category'] != '0']
              .drop_duplicates('product_subcategory')
              .set_index('product_subcategory')['product_category'])
m = df['product_category'] == '0'
df.loc[m, 'product_category'] = df.loc[m, 'product_subcategory'].map(cat_lookup)

#client city has 0 or null fix
df['client_city'] = df['client_city'].str.strip().replace('0', 'Unknown')


In [212]:
rejects_qty = df[df['quantity'] <= 0].copy()  # record of what i removed
df = df[df['quantity'] > 0]

# also remove missing price / product_category
bad_mask = df['unit_price'].isna() | df['product_category'].isna() | (df['product_category'] == '0')
rejects_missing = df[bad_mask].copy()
df = df[~bad_mask]

print('min quantity:', df['quantity'].min())
print('rows:', len(df))
print('quarantined (qty):', len(rejects_qty))
print('quarantined (missing):', len(rejects_missing))

min quantity: 1
rows: 2498
quarantined (qty): 0
quarantined (missing): 0


In [213]:
rejects_qty = pd.read_csv("/content/drive/MyDrive/Exceltis/Exeltis_Data_Engineer_Business_Case_Raw_Data.csv")
rejects_qty = rejects_qty[rejects_qty['quantity'] <= 0]
print(rejects_qty)

   order_id      order_date customer_name client_segment client_city country  \
22   O00023  9/16/2023 0:00      Chen Inc    Distributor  Barnesport   Spain   

   client_singup_date     product_name  quantity  unit_price  unit_cost  \
22              45688  Milk Product 61       -10       26.87      12.92   

    discount_pct product_category product_subcategory           supplier_name  \
22          0.34            Dairy                Milk  Smith, Graham and Rowe   

   product_active sales_channel       sales_rep  
22           True         Phone  Lindsey Curtis  


In [214]:
# quarantine rows missing price or product_category
bad_mask = df['unit_price'].isna() | df['product_category'].isna() | (df['product_category'] == '0')

# add to existing quarantine (or create it)
quarantine = pd.concat([quarantine, df[bad_mask]], ignore_index=True)

# remove from main data
df = df[~bad_mask].reset_index(drop=True)

print('Quarantined rows:', bad_mask.sum())
print('Remaining rows:', len(df))

NameError: name 'quarantine' is not defined

In [ ]:
assert df['order_id'].is_unique, "order_id must be unique"
assert (df['order_date'].notna() & (df['order_date'] != '0')).all(), "no '0' or null in order_date"
assert (df['quantity']>0).all(), "quantity must be positive"
assert df['discount_pct'].dropna().between(0,1).all(), "discount must be 0-1"
assert (df['country'].notna() & (df['country'] != '0')).all(), "no '0' or null in country"
assert (df['client_city'].notna() & (df['client_city'] != '0')).all(), "no '0' or null in client_city"
assert (df['product_category'].notna() & (df['product_category'] != '0')).all(), "no '0' or null in product_category"
assert (df['client_segment'] != '0').all(), "no '0' placeholders in segments"
assert (df['unit_price'] > 0).all(), "unit price must be positive"
assert df['order_date'].dtype == 'datetime64[ns]', "order_date must be datetime"
assert df['client_singup_date'].dtype == 'datetime64[ns]', "client_singup_date must be datetime"

print(" All validation checks passed -", len(df), "clean rows")

In [ ]:
df['order_date'] = pd.to_datetime(df['order_date'], format='%Y/%m/%d')
df['client_singup_date'] = pd.to_datetime(df['client_singup_date'])
print(df['order_date'].dtype)
print(df['order_date'].head())
print(df['client_singup_date'].dtype)
print(df['client_singup_date'].head())

In [ ]:
print('df rows:', len(df))
print('unique customers:',df['customer_name'].nunique())
print('dim_customer rows:', len(dim_customer))
print('dim_customer rows:', len(dim_customer))
print('columns:', df.columns.tolist())

In [ ]:
# dim customer - one row per unique customer
cust_cols = ['customer_name', 'client_segment', 'client_city', 'country', 'client_singup_date']
dim_customer = (
    df[cust_cols]
    .drop_duplicates(subset=['customer_name'])  # one row per customer
    .reset_index(drop=True)
)

dim_customer['client_singup_date'] = dim_customer['client_singup_date'].dt.normalize()
dim_customer['customer_key'] = dim_customer.index + 1

print('dim_customer rows:', len(dim_customer))
dim_customer.head()

In [ ]:
# ==== dim_product
prod_cols = ['product_name','product_category', 'product_subcategory','supplier_name','product_active','unit_cost']
dim_product = (
    df[prod_cols]
    .drop_duplicates(subset=['product_name'])  # one row per product
    .reset_index(drop=True)
)

dim_product['product_key'] = dim_product.index + 1
print('dim_product rows:',len(dim_product))

# ==== dim_channel
dim_channel = (
    df[['sales_channel']]
    .drop_duplicates()
    .reset_index(drop=True)
)

dim_channel['channel_key'] = dim_channel.index + 1
print('dim_channel rows:',len(dim_channel))
dim_product.head()

#dim_date
dim_date = (
    df[['order_date']]
    .drop_duplicates()
    .reset_index(drop=True)
)
dim_date['date_key'] = dim_date.index + 1
dim_date['year'] = dim_date['order_date'].dt.year
dim_date['quarter'] = dim_date['order_date'].dt.quarter
dim_date['month'] = dim_date['order_date'].dt.month
dim_date['month_name'] = dim_date['order_date'].dt.month_name()
dim_date['day_name'] = dim_date['order_date'].dt.day_name()
print('dim_date rows:',len(dim_date))
dim_date.head()


In [ ]:
# Sales the transaction
fact_sales = df.copy()

# join each table to bring in keys
fact_sales = fact_sales.merge(dim_customer[['customer_name', 'customer_key']], on='customer_name', how='left')
fact_sales = fact_sales.merge(dim_product[['product_name', 'product_key']], on='product_name', how='left')
fact_sales = fact_sales.merge(dim_channel[['sales_channel', 'channel_key']], on='sales_channel', how='left')
fact_sales = fact_sales.merge(dim_date[['order_date', 'date_key']], on='order_date', how='left')

#compute the measures
fact_sales['revenue'] = fact_sales['quantity'] * fact_sales['unit_price'] *(1 - fact_sales['discount_pct'].fillna(0))
fact_sales['cost'] = fact_sales['quantity'] * fact_sales['unit_cost'].round(2)
fact_sales['profit'] = fact_sales['revenue'] - fact_sales['cost'].round(2)

#order_id + keys+transactions measures
fact_cols = ['order_id', 'order_date', 'date_key', 'customer_key', 'product_key', 'channel_key', 'quantity', 'unit_price', 'unit_cost', 'discount_pct', 'revenue', 'cost', 'profit']
fact_sales = fact_sales[fact_cols]

print('fact_sales rows:',len(fact_sales))
fact_sales

In [ ]:
#export to csv
fact_sales.to_csv('tbl_sales.csv', index=False)
dim_customer.to_csv('tbl_customer.csv', index=False)
dim_product.to_csv('tbl_product.csv', index=False)
dim_channel.to_csv('tbl_channel.csv', index=False)
dim_date.to_csv('tbl_date.csv', index=False)

# quarantined data into csv (re-read raw to capture original derived values)
rejects = pd.read_csv("/content/drive/MyDrive/Exceltis/Exeltis_Data_Engineer_Business_Case_Raw_Data.csv")
rejects = rejects[
    (rejects['quantity'] <= 0)
    | rejects['unit_price'].isna()
    | rejects['product_category'].isna()
    | (rejects['product_category'] == '0')
]
print(len(rejects), "quarantined row(s)")
rejects.to_csv('tbl_rejects_quarantined.csv', index=False)

print('all exported')
